# Gas Concentration Regression

Estimates analyte concentration (ppm) per gas type from the S11 spectra.

SVR, GB, and GP regressors on PCA features, with data augmentation (noise
injection) applied during training.

In [ ]:
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import WhiteKernel, RBF
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score,
)

warnings.filterwarnings('ignore')

# paths
FILE_PATH    = 'processed_data.pkl'
PRELIM_PATH  = 'preliminary_results.pkl'
RESULTS_PATH = 'regression_results.pkl'
HP_PATH      = 'hyperparams_gas.pkl'

# training flag
SKIP_TRAINING = False   # True -> load hyperparams_gas.pkl, skip Optuna

# constants
RANDOM_STATE     = 42
N_OPTUNA_TRIALS  = 100
N_BOOTSTRAP      = 1000
KFOLD_N_SPLITS   = 5
TUNE_SUBSET_SIZE = 500   # samples per gas used for HP search

# override preliminary settings (None -> use preliminary recommendation)
OVERRIDE_REPR       = 'full'
OVERRIDE_COMPONENTS = 10
OVERRIDE_NOISE      = 0.025

MODEL_NAMES = ['PCA + SVR', 'PCA + GB', 'PCA + GP']

plt.rcParams.update({
    'font.size'         : 10,
    'figure.figsize'    : (14, 5),
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

## Load Data

In [ ]:
with open(FILE_PATH,   'rb') as f: df_all = pickle.load(f)
with open(PRELIM_PATH, 'rb') as f: prelim = pickle.load(f)

df = df_all[df_all['concentration_ul'] > 0].copy().reset_index(drop=True)

GAS_TYPES = sorted(df['gas_type'].unique())

print(f'Sweeps (non-zero conc): {len(df):,}')
print(f'Gas types ({len(GAS_TYPES)}): {GAS_TYPES}')
print()
print('Per-gas counts:')
print(df['gas_type'].value_counts().sort_index().to_string())
print(prelim.keys())

In [ ]:
SENSOR_NAMES = [
    'Sensor A — GO/Nafion (S11)',
    'Sensor B — G/GO/PEDOT:PSS (S22)',
]

def _resolve(override, prelim_val):
    return override if override is not None else prelim_val

SENSOR_CONFIG = {
    SENSOR_NAMES[0]: {
        'feature_col' : 'features_a',
        'best_repr'   : _resolve(OVERRIDE_REPR,       prelim[SENSOR_NAMES[0]]['best_repr']),
        'n_components': _resolve(OVERRIDE_COMPONENTS, prelim[SENSOR_NAMES[0]]['n_components']),
    },
    SENSOR_NAMES[1]: {
        'feature_col' : 'features_b',
        'best_repr'   : _resolve(OVERRIDE_REPR,       prelim[SENSOR_NAMES[1]]['best_repr']),
        'n_components': _resolve(OVERRIDE_COMPONENTS, prelim[SENSOR_NAMES[1]]['n_components']),
    },
}
NOISE_LEVEL = _resolve(OVERRIDE_NOISE, prelim['noise_level'])

for sn, cfg in SENSOR_CONFIG.items():
    print(f'  {sn}: repr={cfg["best_repr"]}, n_comp={cfg["n_components"]}')
print(f'  NOISE_LEVEL = {NOISE_LEVEL}')

## Feature Representation

In [ ]:
N_FREQ_PTS = len(df['features_a'].iloc[0])
_i1 = int(N_FREQ_PTS * (6000 - 2000) / (8000 - 2000))   # 2-6 GHz upper index


def build_representation(X_raw: np.ndarray, repr_name: str) -> np.ndarray:
    if repr_name == 'full':
        return X_raw
    elif repr_name == 'band_limited':
        return X_raw[:, :_i1]
    elif repr_name == 'derivative':
        return np.gradient(X_raw, axis=1)
    else:
        raise ValueError(f"Unknown representation '{repr_name}'")

## Helper Functions

In [ ]:
# subset sampling

def random_subset_idx(n_total, n_subset, rng):
    """Random subset index for HP tuning on a small fraction of the gas data."""
    return rng.choice(n_total, size=min(n_subset, n_total), replace=False)


# inner-fold pre-computation

def precompute_inner_kfold_folds(X, y, n_components):
    """Pre-bake noise / scaling / PCA for each KFold inner fold."""
    folded = []
    kf = KFold(n_splits=KFOLD_N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for tr, te in kf.split(X):
        Xtr = X[tr] + np.random.normal(0, NOISE_LEVEL, X[tr].shape)
        sc  = StandardScaler().fit(Xtr)
        pca = PCA(n_components=n_components, random_state=RANDOM_STATE).fit(sc.transform(Xtr))
        folded.append((
            pca.transform(sc.transform(Xtr)), y[tr],
            pca.transform(sc.transform(X[te])), y[te],
        ))
    return folded


# Optuna objective

def make_objective(model_name, precomputed_folds):
    def objective(trial):
        if model_name == 'PCA + SVR':
            reg = SVR(
                kernel='rbf', max_iter=50000,
                C      = trial.suggest_float('C',       1e-1, 1e3, log=True),
                epsilon= trial.suggest_float('epsilon', 1e-3, 1.0, log=True),
                gamma  = trial.suggest_categorical('gamma', ['scale', 'auto']),
            )
        elif model_name == 'PCA + GB':
            reg = GradientBoostingRegressor(
                random_state     = RANDOM_STATE,
                n_estimators     = trial.suggest_int  ('n_estimators',     50,   500),
                learning_rate    = trial.suggest_float('learning_rate',    0.01, 0.3, log=True),
                max_depth        = trial.suggest_int  ('max_depth',        2,    6),
                subsample        = trial.suggest_float('subsample',        0.5,  1.0),
                min_samples_leaf = trial.suggest_int  ('min_samples_leaf', 1,    20),
            )
        else:
            raise ValueError(model_name)
        return float(np.mean([
            mean_absolute_error(yte, reg.fit(Xtr, ytr).predict(Xte))
            for Xtr, ytr, Xte, yte in precomputed_folds
        ]))
    return objective


def run_optuna(model_name, inner_folds):
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(
        make_objective(model_name, inner_folds),
        n_trials=N_OPTUNA_TRIALS, show_progress_bar=True, n_jobs=-1,
    )
    return study.best_params, study.best_value


# pipeline builder

def build_tuned_pipeline(model_name, best_params, n_components):
    if model_name == 'PCA + SVR':
        reg = SVR(kernel='rbf', max_iter=50000, **best_params)
    elif model_name == 'PCA + GB':
        reg = GradientBoostingRegressor(random_state=RANDOM_STATE, **best_params)
    elif model_name == 'PCA + GP':
        reg = GaussianProcessRegressor(
            kernel=1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e3))
                   + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e2)),
            normalize_y=True, n_restarts_optimizer=3, random_state=RANDOM_STATE,
        )
    else:
        raise ValueError(model_name)
    return Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=n_components, random_state=RANDOM_STATE)),
        ('reg',    reg),
    ])


# bootstrap CI

def bootstrap_ci(y_true, y_pred, n_bootstrap=N_BOOTSTRAP, ci=0.95):
    rng   = np.random.default_rng(RANDOM_STATE)
    n     = len(y_true)
    mae_b, r2_b = [], []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        mae_b.append(mean_absolute_error(y_true[idx], y_pred[idx]))
        r2_b.append (r2_score          (y_true[idx], y_pred[idx]))
    alpha = (1 - ci) / 2
    return {
        'mae': (np.mean(mae_b),
                np.percentile(mae_b, alpha * 100),
                np.percentile(mae_b, (1-alpha) * 100)),
        'r2' : (np.mean(r2_b),
                np.percentile(r2_b,  alpha * 100),
                np.percentile(r2_b,  (1-alpha) * 100)),
    }


# binary classification (derived)

def binary_classification_metrics(y_true, y_pred, threshold):
    """Derive binary high/low-concentration task from regression predictions."""
    y_true_bin = (y_true > threshold).astype(int)
    y_pred_bin = (y_pred > threshold).astype(int)
    return {
        'accuracy'  : accuracy_score (y_true_bin, y_pred_bin),
        'precision' : precision_score(y_true_bin, y_pred_bin, zero_division=0),
        'recall'    : recall_score   (y_true_bin, y_pred_bin, zero_division=0),
        'f1'        : f1_score       (y_true_bin, y_pred_bin, zero_division=0),
        'roc_auc'   : roc_auc_score  (y_true_bin, y_pred),
        'y_true_bin': y_true_bin,
        'y_score'   : y_pred,
        'threshold' : threshold,
    }


print('Helper functions defined.')

## Hyperparameter Tuning

Optuna minimises mean MAE over 5 inner K-Fold
folds on a random 500-sample subset. GP uses sklearn's marginal-likelihood
kernel optimiser instead of Optuna.

In [ ]:
rng    = np.random.default_rng(RANDOM_STATE)
hp_gas = {}   # {sensor: {gas_type: {model: params}}}

if SKIP_TRAINING:
    with open(HP_PATH, 'rb') as f:
        hp_gas = pickle.load(f)
    print(f'Loaded cached hyperparams from {HP_PATH}')
else:
    for sensor_name, cfg in SENSOR_CONFIG.items():
        print(f'\n── {sensor_name} ──')
        hp_gas[sensor_name] = {}

        X_raw = np.stack(df[cfg['feature_col']].values)
        X     = build_representation(X_raw, cfg['best_repr'])
        y     = df['concentration_ppm'].values
        gas   = df['gas_type'].values
        nc    = cfg['n_components']

        for gas_type in GAS_TYPES:
            print(f'\n  [{gas_type}]')
            mask       = gas == gas_type
            X_g, y_g   = X[mask], y[mask]

            sub         = random_subset_idx(len(X_g), TUNE_SUBSET_SIZE, rng)
            inner_folds = precompute_inner_kfold_folds(X_g[sub], y_g[sub], nc)
            print(f'    Subset: {len(sub)} samples | {len(inner_folds)} inner folds')

            hp_gas[sensor_name][gas_type] = {}
            for model_name in ['PCA + SVR', 'PCA + GB']:
                print(f'    Tuning {model_name} ({N_OPTUNA_TRIALS} trials)...')
                best_params, best_val = run_optuna(model_name, inner_folds)
                hp_gas[sensor_name][gas_type][model_name] = best_params
                print(f'      best inner MAE = {best_val:.2f} ppm  |  {best_params}')

            hp_gas[sensor_name][gas_type]['PCA + GP'] = {}

    with open(HP_PATH, 'wb') as f:
        pickle.dump(hp_gas, f)
    print(f'\nSaved → {HP_PATH}')

## K-Fold Cross-Validation

5-fold random K-Fold per gas type, using the tuned hyperparameters.

In [ ]:
all_results = {}   # {sensor: {gas_type: {model: results}}}

for sensor_name, cfg in SENSOR_CONFIG.items():
    print(f'\n{"="*60}\n{sensor_name}\n{"="*60}')
    all_results[sensor_name] = {}

    X_raw = np.stack(df[cfg['feature_col']].values)
    X     = build_representation(X_raw, cfg['best_repr'])
    y     = df['concentration_ppm'].values
    gas   = df['gas_type'].values
    nc    = cfg['n_components']
    kf    = KFold(n_splits=KFOLD_N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for gas_type in GAS_TYPES:
        mask     = gas == gas_type
        X_g, y_g = X[mask], y[mask]
        all_results[sensor_name][gas_type] = {}

        for model_name in MODEL_NAMES:
            pipe = build_tuned_pipeline(
                model_name, hp_gas[sensor_name][gas_type][model_name], nc)

            fold_y_true, fold_y_pred = [], []
            for tr_idx, te_idx in kf.split(X_g):
                X_tr = X_g[tr_idx] + np.random.normal(0, NOISE_LEVEL, X_g[tr_idx].shape)
                pipe.fit(X_tr, y_g[tr_idx])
                fold_y_true.append(y_g[te_idx])
                fold_y_pred.append(pipe.predict(X_g[te_idx]))

            y_true_all = np.concatenate(fold_y_true)
            y_pred_all = np.concatenate(fold_y_pred)
            threshold  = float(np.median(y_true_all))

            ci_dict = bootstrap_ci(y_true_all, y_pred_all)
            binary  = binary_classification_metrics(y_true_all, y_pred_all, threshold)
            pr, _   = pearsonr (y_true_all, y_pred_all)
            sr, _   = spearmanr(y_true_all, y_pred_all)

            all_results[sensor_name][gas_type][model_name] = {
                'y_true'      : y_true_all,
                'y_pred'      : y_pred_all,
                'r2'          : r2_score(y_true_all, y_pred_all),
                'mae'         : mean_absolute_error(y_true_all, y_pred_all),
                'rmse'        : float(np.sqrt(mean_squared_error(y_true_all, y_pred_all))),
                'pearson_r'   : pr,
                'spearman_rho': sr,
                'ci_mae'      : ci_dict['mae'],
                'ci_r2'       : ci_dict['r2'],
                'binary'      : binary,
                'threshold'   : threshold,
            }
            res = all_results[sensor_name][gas_type][model_name]
            print(f'  [{gas_type:8s}] [{model_name:10s}]  '
                  f'R²={res["r2"]:.3f}  MAE={res["mae"]:.1f} ppm  '
                  f'AUC={binary["roc_auc"]:.3f}')

print('\nEvaluation complete.')

In [ ]:
reg_rows, clf_rows = [], []

for sensor_name, gas_results in all_results.items():
    slabel = sensor_name.split('—')[0].strip()
    for gas_type, model_results in gas_results.items():
        for model_name, res in model_results.items():
            reg_rows.append({
                'Sensor'     : slabel,
                'Gas'        : gas_type,
                'Model'      : model_name,
                'R2'         : round(res['r2'],           4),
                'MAE (ppm)'  : round(res['mae'],          2),
                'RMSE (ppm)' : round(res['rmse'],         2),
                'Pearson r'  : round(res['pearson_r'],    4),
                'Spearman ρ' : round(res['spearman_rho'], 4),
                'CI MAE lo'  : round(res['ci_mae'][1],    2),
                'CI MAE hi'  : round(res['ci_mae'][2],    2),
                'CI R2 lo'   : round(res['ci_r2'][1],     4),
                'CI R2 hi'   : round(res['ci_r2'][2],     4),
            })
            b = res['binary']
            clf_rows.append({
                'Sensor'         : slabel,
                'Gas'            : gas_type,
                'Model'          : model_name,
                'Threshold (ppm)': round(res['threshold'], 1),
                'Accuracy'       : round(b['accuracy'],  4),
                'Precision'      : round(b['precision'], 4),
                'Recall'         : round(b['recall'],    4),
                'F1'             : round(b['f1'],        4),
                'ROC-AUC'        : round(b['roc_auc'],   4),
            })

df_reg = pd.DataFrame(reg_rows)
df_clf = pd.DataFrame(clf_rows)

print('=== Regression (sample — first gas) ===')
print(df_reg[df_reg['Gas'] == GAS_TYPES[0]].to_string(index=False))
print('\n=== Binary Classification derived (sample — first gas) ===')
print(df_clf[df_clf['Gas'] == GAS_TYPES[0]].to_string(index=False))

## Export Results

In [ ]:
results_bundle = {
    'cv_results'    : all_results,      # {sensor: {gas_type: {model: results}}}
    'hyperparams'   : hp_gas,           # {sensor: {gas_type: {model: params}}}
    'regression_df' : df_reg,
    'clf_df'        : df_clf,
    'sensor_config' : SENSOR_CONFIG,
    'gas_types'     : GAS_TYPES,
    'metadata': {
        'noise_level'      : NOISE_LEVEL,
        'n_optuna_trials'  : N_OPTUNA_TRIALS,
        'n_bootstrap'      : N_BOOTSTRAP,
        'kfold_n_splits'   : KFOLD_N_SPLITS,
        'tune_subset_size' : TUNE_SUBSET_SIZE,
        'source_file'      : FILE_PATH,
        'prelim_file'      : PRELIM_PATH,
        'model_names'      : MODEL_NAMES,
        'cv_scheme'        : 'KFold per gas (5-fold random)',
        'target'           : 'concentration_ppm',
        'task'             : 'per-gas concentration regression',
    },
}

with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(results_bundle, f)

print(f'Saved {RESULTS_PATH}')
print(f'Saved {HP_PATH} (already written during tuning)')
print('Ready for 01_results.ipynb')